# 本地 LLM Vision（OpenAI 兼容，例如 LM Studio）

- 服务根地址：`http://127.0.0.1:1234`；Python 里请把 **base_url** 设为 `http://127.0.0.1:1234/v1`（多出来的 `/v1` 是 OpenAI 兼容路由）。
- 在 LM Studio 中需**加载支持视觉的多模态模型**；若未设置环境变量 `LOCAL_VISION_MODEL`，脚本会自动选用当前服务上的第一个模型。
- 示例从本地文件读入图片，再编码为 base64 放入 `data:` URL（不要把文件路径当作 base64 拼进 URL）。

In [7]:
%pip install -q openai


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from openai import OpenAI

# 服务根地址一般是 http://127.0.0.1:1234 ，OpenAI 兼容接口需在路径上加 /v1
BASE_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL = "qwen/qwen3-vl-4b"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# 未配置 LOCAL_VISION_MODEL 时，自动用当前服务上的第一个模型（LM Studio 通常只加载一个）
if MODEL == "replace-with-your-model-id":
    models = client.models.list().data
    if not models:
        raise RuntimeError("本地服务未返回任何模型，请先在 LM Studio 等中加载模型。")
    MODEL = models[0].id
    print("使用模型:", MODEL)

In [9]:
# 可选：列出当前服务上的模型 id，把上一单元里的 MODEL 改成其中之一（须为支持 vision 的模型）
for m in client.models.list().data:
    print(m.id)

qwen/qwen3-vl-4b
liquid/lfm2.5-1.2b
google/gemma-4-e4b
deepseek/deepseek-r1-0528-qwen3-8b
openai_gpt-oss-20b_pruned_reap_10b
text-embedding-nomic-embed-text-v1.5


In [10]:
# 读本地图片：读入字节后做 base64，再放进 data: URL（不要把「文件路径」当成 base64 字符串）
import base64
from pathlib import Path

IMAGE_NAME = "微信图片_20260504122723_232_510.jpg"


def resolve_local_image(name: str) -> Path:
    """兼容 cwd 为仓库根目录或 notebooks/ 等情况。"""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for rel in (
            base / "notebooks" / "19_image" / name,
            base / "19_image" / name,
        ):
            if rel.is_file():
                return rel.resolve()
    raise FileNotFoundError(
        f"找不到 {name}。当前 cwd: {here}。请确认文件在 notebooks/19_image/ 下，或把 IMAGE_PATH 改成绝对路径。"
    )


IMAGE_PATH = resolve_local_image(IMAGE_NAME)
print("使用图片:", IMAGE_PATH)

suffix = IMAGE_PATH.suffix.lower()
mime = "image/jpeg" if suffix in {".jpg", ".jpeg"} else "image/png" if suffix == ".png" else "application/octet-stream"

b64 = base64.standard_b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
image_url = f"data:{mime};base64,{b64}"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请用一句话描述这张图片（若看不清则说明即可）。"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    max_tokens=256,
)

print(resp.choices[0].message.content)

FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/19_image/微信图片_20260504122723_232_510.jpg'